# Внутренний аудит закупочной деятельности группы Сбер (2024–2025)

В данном Jupyter Notebook реализована техническая часть проекта: очистка сырых данных, дедупликация, нормализация и подготовка датасета для заливки в СУБД PostgreSQL.

> **Важно: Ограничения выборки и методология**
> 
> Подробное описание того, как собирались данные, почему был выбран именно ИТ-контур из 10 компаний (а не головной ПАО Сбербанк) и как на сбор повлияли **Постановление Правительства РФ №301** и **закрытие открытого доступа к API ЕИС с 1 января 2025 года**, находится в главном файле проекта — `README.md`.
>
> Рекомендую начать ознакомление с проектом именно с него!

Ниже приступаем непосредственно к обработке собранного массива из 1 859 записей.

In [4]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## Этап 1. Сбор данных

**Описание источников и обоснование выбора:**
Сбор данных проводился с коммерческой секции SberB2B (ЭТП Сбербанк-АСТ). 
Почему не официальная часть ЕИС:
1. **ПП РФ №301 от 06.03.2022:** Головной ПАО Сбербанк освобожден от обязательной публикации закупок в открытой части ЕИС из-за санкций.
2. **Ограничения API:** С 1 января 2025 года ЕИС закрыл массовую выгрузку по открытому протоколу. 
Поэтому фокус исследования смещен на ИТ/финтех-контур дочерних обществ Сбера (ДЗО), данные по которым собирались инструментами парсинга.

**Составление перечня юридических лиц:**
Для анализа отобран ИТ-контур из 10 компаний (СберТех, СберСервис, СалютДевайсы и др.).

**Обезличивание персональных данных:**
Поскольку анализируются b2b-закупки крупных дочерних обществ, в собранном массиве данных отсутствуют ФИО физических лиц и паспортные данные. Обезличивание не потребовалось.

In [5]:
list_faces = pd.read_excel('../data/list_of_entities.xlsx')
list_faces.head(10)

,Название компании,Сфера деятельности,Юридический статус / Бывшее название,ИНН,УНП
0,ПАО Сбербанк,Банкинг и финансы,Головная организация,7707083893.00,NaN
1,ООО «Драйв Клик Банк»,Банкинг и финансы,Дочерний банк (автокредитование) / Сетелем Банк,6452010742.00,NaN
2,ОАО «Сбер Банк» (Беларусь),Банкинг и финансы,Дочерний банк в Республике Беларусь,NaN,100219673.00
3,АО «НПФ Сбербанка»,Банкинг и финансы,Негосударственный пенсионный фонд,7725352740.00,NaN
4,ООО СК «Сбербанк Страхование»,Банкинг и финансы,Страховая компания,7706810747.00,NaN
5,ООО СК «Сбербанк Страхование жизни»,Банкинг и финансы,Страховая компания холдинга,7706404944.00,NaN
6,АО «Сбербанк Лизинг»,Банкинг и финансы,Лизинговые операции,7707009586.00,NaN
7,ООО «Сбербанк Факторинг»,Банкинг и финансы,Факторинговые операции,7802754982.00,NaN
8,ООО «Сбербанк Капитал»,Банкинг и финансы,Управление инвестициями и активами,7707670104.00,NaN
9,ООО «Сбербанк Инвестиции»,Банкинг и финансы,Управление инвестициями и санация,7707446480.00,NaN


## Этап 2. Обработка и База данных

**Очистка данных и статистика по дублям:**
При выгрузке данных с площадки возникают дубликаты из-за обновления статусов карточек (например, переход из «Подача заявок» в «Архив»). Производится дедупликация по ключу `Код процедуры`.

In [6]:
tenders = pd.read_excel('../data/sber_audit_converted.xlsx')

tenders = tenders.drop_duplicates(subset=['Код процедуры'], keep='last')

print(f"До очистки: {len(pd.read_excel('../data/sber_audit_converted.xlsx'))} записей")
print(f"После очистки: {len(tenders)} записей")
print(f"Удалено дублей: {len(pd.read_excel('../data/sber_audit_converted.xlsx')) - len(tenders)}")


До очистки: 1859 записей
После очистки: 1364 записей
Удалено дублей: 495


#### Обогащение данных

In [7]:
inn_mapping = {
    'АО «Сбербанк Лизинг»': '7707009586',
    'АО «СберТех»': '7736632467',
    'ООО «СалютДевайсы»': '7730253720',
    'ООО «ЮМани»': '7750005725',
    'ООО «Облачные технологии»': '7736279160',
    'ООО «Звук»': '7708328948',
    'АО «СберСервис»': '7736663049',
    'ООО «С-Маркетинг»': '7736319695',
    'АО «ОКБ»': '7710561081',
    'АО «Деловая среда»': '7736641983'
}
tenders['ИНН_Заказчика'] = tenders['Компания-заказчик'].map(inn_mapping)

tenders['Начальная сумма'] = (
    tenders['Начальная сумма']
    .astype(str)
    .str.replace('\xa0', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

**Работа с приложенными документами (Концепция):**
В рамках собранного массива присутствуют ссылки на тендерную документацию. Идеи для их дальнейшего применения в аналитике:
1. **Парсинг ТЗ с помощью LLM:** Автоматическое извлечение сроков поставки, штрафных санкций (SLA) и требуемых брендов оборудования.
2. **Анализ контрактов:** Проверка проектов договоров на аномальные авансовые платежи (риск вывода средств).

**Проектирование БД PostgreSQL и реализация SQL-запросов:**
Целевая нормализованная база спроектирована в PostgreSQL (DDL-схема доступна в `sql/schema.sql`).

> Изначально БД и запросы были написаны для PostgreSQL, но для обеспечения воспроизводимости процесса проверки аналитического модуля ниже разворачивается
> in-memory БД `SQLite`, куда загружается плоская витрина очищенных данных для демонстрации SQL-запросов.

In [8]:
# Для удобства написания запросов
rename_dict = {
    'Компания-заказчик': 'company_name',
    'Способ закупки': 'method_name',
    'Код процедуры': 'procedure_code',
    'Наименование предмета закупки': 'title',
    'Начальная сумма': 'initial_amount',
    'Дата публикации': 'published_at',
    'Окончание подачи заявок': 'deadline_at',
    'Тип закупки': 'type_name'
}
tenders = tenders.rename(columns=rename_dict)

conn = sqlite3.connect(':memory:')
tenders.to_sql('procurement_flat', conn, if_exists='replace', index=False)

1364

### Базовые и комплаенс-проверки данных (SQL)
Для выявления аномалий и подготовки данных к визуализации реализовано 5 аналитических запросов:
1. Сверка полноты миграции данных по ДЗО.
2. Поиск скрытых/технических закупок с НМЦ = 0 руб.
3. Оценка доли неконкурентных способов закупки (через подзапросы).
4. Комплаенс-риски: Выявление процедур >10 млн руб со сроком подачи заявок менее 5 дней.
5. Профилирование бюджета по ИТ-категориям.

In [ ]:
q1 = """
SELECT company_name, COUNT(procedure_code) AS procedure_count, SUM(initial_amount) AS budget
FROM procurement_flat GROUP BY company_name ORDER BY budget DESC;
"""
print("--- 1. Сверка миграции данных ---")
display(pd.read_sql_query(q1, conn))

q2 = """
SELECT procedure_code, company_name, title, published_at
FROM procurement_flat WHERE initial_amount = 0.00 ORDER BY published_at DESC;
"""
print("\n--- 2. Закупки с нулевой стоимостью ---")
display(pd.read_sql_query(q2, conn))

q3 = """
SELECT method_name AS method, COUNT(procedure_code) AS count_procedure,
       ROUND(COUNT(procedure_code) * 100.0 / (SELECT COUNT(*) FROM procurement_flat), 2) AS pct_by_count,
       SUM(initial_amount) AS budget,
       ROUND(SUM(initial_amount) * 100.0 / (SELECT SUM(initial_amount) FROM procurement_flat), 2) AS pct_by_budget
FROM procurement_flat GROUP BY method_name ORDER BY budget DESC;
"""
print("\n--- 3. Анализ способов закупки ---")
display(pd.read_sql_query(q3, conn))

q4 = """
SELECT procedure_code, company_name, title, initial_amount AS sum,
       CAST((julianday(deadline_at) - julianday(published_at)) AS INTEGER) AS submission_days
FROM procurement_flat
WHERE submission_days < 5 AND initial_amount > 10000000.00
ORDER BY submission_days ASC, sum DESC;
"""
print("\n--- 4. Комплаенс-риски: Срочные дорогие закупки ---")
display(pd.read_sql_query(q4, conn))

q5 = """
SELECT type_name AS category_name, COUNT(procedure_code) AS count_procedure, 
       SUM(initial_amount) AS budget, AVG(initial_amount) AS average_check
FROM procurement_flat GROUP BY type_name ORDER BY budget DESC;
"""
print("\n--- 5. Профилирование по категориям ---")
display(pd.read_sql_query(q5, conn))